In [1]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pathlib import Path

# 1. Define your paths
raster_dir = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\rasters")
# Now pointing to a single master shapefile instead of a directory
master_shapefile_path = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\AOI\FAO_RCP51_Country.shp") 
output_dir = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\clipped_output")

output_dir.mkdir(parents=True, exist_ok=True)

# 2. Load the master shapefile ONCE (saves time and memory)
print("Loading master shapefile...")
sea_gdf = gpd.read_file(master_shapefile_path)

# 3. Iterate through all raster files in the directory
for raster_path in raster_dir.glob("*.tif"): 
    
    # Extract country code from filename: palm_binary_KHM_2024 -> KHM
    # name.split('_') = ['palm', 'binary', 'KHM', '2024']
    filename_parts = raster_path.stem.split('_')
    
    # Safety check: ensure the filename has enough parts
    if len(filename_parts) < 3:
        print(f"Skipping {raster_path.name}: Filename format unrecognized.")
        continue
        
    country_code = filename_parts[2] 
    
    print(f"Processing {country_code}...")

    # 4. Filter the master shapefile for the specific country
    # This selects only the row(s) where the ISO_A3 column matches our country code
    country_gdf = sea_gdf[sea_gdf['ISO_A3'] == country_code]
    
    if country_gdf.empty:
        print(f"  Warning: No features found for {country_code} in the shapefile. Skipping...")
        continue

    # 5. Open the raster file
    with rasterio.open(raster_path) as src:
        
        # 6. Ensure CRS match for the specific country slice
        if country_gdf.crs != src.crs:
            print(f"  Reprojecting {country_code} boundary to match raster CRS...")
            country_gdf = country_gdf.to_crs(src.crs)
            
        # Extract the geometries as a list for rasterio
        geometries = country_gdf.geometry.tolist()
        
        try:
            # 7. Apply the mask
            out_image, out_transform = mask(src, geometries, crop=True)
            
            # 8. Update metadata
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform
            })
            
            # 9. Save the clipped raster
            out_file = output_dir / f"{raster_path.stem}_clipped.tif"
            with rasterio.open(out_file, "w", **out_meta) as dest:
                dest.write(out_image)
                
            print(f"  Successfully saved: {out_file.name}")
            
        except ValueError as e:
            print(f"  Error masking {country_code} ({raster_path.name}): {e}")

print("Batch clipping complete.")

Loading master shapefile...
Processing BTN...
  Reprojecting BTN boundary to match raster CRS...
  Successfully saved: cocoa_binary_BTN_2024_clipped.tif
Processing KHM...
  Reprojecting KHM boundary to match raster CRS...
  Successfully saved: cocoa_binary_KHM_2024_clipped.tif
Processing LAO...
  Reprojecting LAO boundary to match raster CRS...
  Successfully saved: cocoa_binary_LAO_2024_clipped.tif
Processing MMR...
  Reprojecting MMR boundary to match raster CRS...
  Successfully saved: cocoa_binary_MMR_2024_clipped.tif
Processing MYS...
  Reprojecting MYS boundary to match raster CRS...
  Successfully saved: cocoa_binary_MYS_2024_clipped.tif
Processing PHL...
  Reprojecting PHL boundary to match raster CRS...
  Successfully saved: cocoa_binary_PHL_2024_clipped.tif
Processing PNG...
  Reprojecting PNG boundary to match raster CRS...
  Successfully saved: cocoa_binary_PNG_2024_clipped.tif
Processing THA...
  Reprojecting THA boundary to match raster CRS...
  Successfully saved: cocoa_

In [2]:
import glob
from rasterio.merge import merge
#Gather all individual country binary rasters
file_list = glob.glob(r"raster_mosaicking\commodities_country\cocoa_clip\*.tif")
src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]
# 2. Merge rasters spatially into a single regional raster
# Using method='max' ensures that if country borders overlap, presence (1) overrides absence (0)
mosaic, out_transform = merge(
    src_files_to_mosaic, 
    method='max', 
    nodata=0
)
# 3. Update metadata to reflect the expanded regional extent and grid resolution
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "nodata": 0,
    "compress": "lzw"
})

IndexError: list index out of range

In [9]:
from pathlib import Path
import glob
import rasterio
from rasterio.merge import merge

folder = Path(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\commodities_country\coffee_clip")
file_list = sorted(str(p) for p in folder.glob("*.tif"))

print("Found files:", file_list)

if not file_list:
    raise FileNotFoundError(f"No TIFF files found in {folder}")

src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]

mosaic, out_transform = merge(src_files_to_mosaic, method="max", nodata=0)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "nodata": 0,
    "compress": "lzw"
})

Found files: ['C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_BTN_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_IDN_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_IDN_KAL_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_KHM_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_LAO_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_country\\coffee_clip\\coffee_binary_MMR_2024_clipped.tif', 'C:\\Users\\AFahrezi\\Documents\\GitHub\\generic_rs_code\\raster_mosaicking\\commodities_count

In [10]:
with rasterio.open(r"C:\Users\AFahrezi\Documents\GitHub\generic_rs_code\raster_mosaicking\commodities_country\coffee_binary_combine_2024.tif", "w", **out_meta) as dest:
    dest.write(mosaic)
#Close open file connections
for src in src_files_to_mosaic:
    src.close()